In [1]:
import re
from pylatexenc.latexwalker import (
    LatexWalker, LatexCharsNode, LatexMacroNode, LatexGroupNode, LatexMathNode
)

# ==========================================
# 1. 制定“海关安检规则”（黑白名单）
# ==========================================

# 黑名单：遇到这些命令，直接“砍断树枝”，绝对不看它包裹的内容！
# 例如遇到下划线 _ ，我们就不要去提取下标里的 i 或 j
skip_children_macros = ['_', '^'] 

# 白名单 A：合法的数学算子。遇到这些连续的英文字母，当成整体保留。
operator_whitelist = ['E', 'Var', 'Cov']

# 白名单 B：合法的希腊字母。遇到这些 LaTeX 命令，加上斜杠保留。
greek_letters = ['alpha', 'beta', 'mu', 'sigma', 'pi', 'delta', 'Delta']

# ==========================================
# 2. 核心提取引擎（你的第一个业务函数）
# ==========================================
def extract_symbols(node_list):
    symbols = set() # 用 set 集合自动去重（如果有三个 x，只记一次）
    
    if not node_list:
        return symbols
        
    for node in node_list:
        # 情况 A：普通纯文本（比如 "E[(X" 或 "2+"）
        if isinstance(node, LatexCharsNode):
            # 用正则把连续的英文字母抠出来
            words = re.findall(r'[A-Za-z]+', node.chars)
            for w in words:
                # 核心规则：如果是单字母 (如 X, y) 或在算子白名单里 (如 E)，就放行！
                if len(w) == 1 or w in operator_whitelist:
                    symbols.add(w)
                    
        # 情况 B：遇到以反斜杠开头的 LaTeX 命令（比如 \sigma, \frac, \_）
        elif isinstance(node, LatexMacroNode):
            macroname = node.macroname
            
            # 如果是希腊字母，立刻收录！
            if macroname in greek_letters:
                symbols.add("\\" + macroname)
                
            # 【核心魔法】：如果是黑名单里的上下标，直接 continue 跳过！
            # 这样就不会把 x_i 里面的 i 当成独立变量提取出来了
            if macroname in skip_children_macros:
                continue 
                
            # 如果不是黑名单，且它包裹了内容（比如 \frac{A}{B} 或 \overline{\delta}）
            # 就继续“钻进去”扒它的皮
            if node.nodeargd and node.nodeargd.argnlist:
                for arg_node in node.nodeargd.argnlist:
                    symbols.update(extract_symbols([arg_node]))
                    
        # 情况 C：遇到花括号组或数学环境，直接钻进去解析
        elif isinstance(node, (LatexGroupNode, LatexMathNode)):
            symbols.update(extract_symbols(node.nodelist))
            
    return symbols

# ==========================================
# 3. 极限压力测试 (用你 Chapter 6 的真实公式)
# ==========================================
print("=== 实验一：算子与过滤黑白名单测试 ===\n")

# 测试 1：\frac 能不能剥皮提取？
form1 = r"\frac{A}{B}"
nodes1 = LatexWalker(form1).get_latex_nodes()[0]
print(f"公式 1: {form1}\n提取结果: {extract_symbols(nodes1)}\n" + "-"*30)

# 测试 2：\sum 和 下标 _i 能不能完美过滤掉？
form2 = r"\sum_{i=1}^n x_i"
nodes2 = LatexWalker(form2).get_latex_nodes()[0]
print(f"公式 2: {form2}\n提取结果: {extract_symbols(nodes2)}\n" + "-"*30)

# 测试 3：Chapter 6 的魔王公式，能不能把期望 E 和希腊字母认出来？
form3 = r"\sigma^2 = E[(X-\mu)^2]"
nodes3 = LatexWalker(form3).get_latex_nodes()[0]
print(f"公式 3: {form3}\n提取结果: {extract_symbols(nodes3)}")

=== 实验一：算子与过滤黑白名单测试 ===

公式 1: \frac{A}{B}
提取结果: {'B', 'A'}
------------------------------
公式 2: \sum_{i=1}^n x_i
提取结果: {'i', 'x', 'n'}
------------------------------
公式 3: \sigma^2 = E[(X-\mu)^2]
提取结果: {'X', '\\sigma', 'E', '\\mu'}


In [6]:
import json
import glob
import re
import os

# 4. 批量运行 chapter6 公式库里的公式
formula_library_path = 'chapter6/formula_library.json'
print(f'公式库路径：{formula_library_path}')

with open(formula_library_path, 'r', encoding='utf-8-sig') as f:
    formula_library = json.load(f)

expressions = [item['latex'] for item in formula_library.get('formulas', []) if item.get('latex')]
print(f'公式库中共有 {len(expressions)} 个公式')

total_formulas = len(expressions)
parsed_total = 0
failed_parses = 0
first_example = None
for expr in expressions:
    try:
        node = LatexWalker(expr).get_latex_nodes()[0]
        parsed_total += 1
        if first_example is None:
            first_example = (expr, extract_symbols(node))
    except Exception:
        failed_parses += 1

print('=== chapter6 公式库解析汇总 ===')
print(f'公式总数：{total_formulas}')
print(f'成功解析公式数：{parsed_total}')
print(f'解析失败公式数：{failed_parses}')
if first_example is not None:
    expr, symbols = first_example
    print('\n第一个公式示例：')
    print(f'公式内容：${expr}$')
    print('提取结果：', symbols)


公式库路径：chapter6/formula_library.json
公式库中共有 82 个公式
=== chapter6 公式库解析汇总 ===
公式总数：82
成功解析公式数：82
解析失败公式数：0

第一个公式示例：
公式内容：$\overline{z}=\sum_{i=1}^{N}q_{i}z_{i}$
提取结果： {'N', 'z', 'i', 'q'}
